# Epsilon experiments on Kaggle (Nkusi Dan)

My laptop is too slow to train on the CPU, so I run my 10 experiments on a Kaggle GPU. Each run trains with `train.py`,
and then `evaluate.py` plays 5 games greedily (epsilon = 0) with the saved model.

Before running: Settings -> Accelerator: GPU T4, and Internet on. Then Save Version -> Save & Run All so it keeps running
with the browser closed. The results are in the Output tab.

In [ ]:
!nvidia-smi

I clone into /tmp so the repo and the .venv don't end up in the notebook output, only the runs do.

In [ ]:
%cd /tmp
!git clone https://github.com/Samkwizera/deep_q_lerning-group-12.git
%cd /tmp/deep_q_lerning-group-12

In [ ]:
import os

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
# kaggle sets this to its inline backend, which the uv venv doesn't have, so matplotlib crashes on import
os.environ["MPLBACKEND"] = "Agg"
!uv sync
# shell commands don't stop the notebook when they fail, so check by hand
if _exit_code != 0:
    raise RuntimeError("uv sync failed, check the output above")
!uv run python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

## Runs

Samuel tuned epsilon, so I tune learning rate, gamma and batch size. Epsilon is fixed at 0.1 -> 0.01 over 50k steps,
the setting from Samuel's exp09, the only one that learned to win. Then the agent learns early enough for the
differences between runs to show. Everything else stays at the `train.py` defaults (seed 42), and every run is 500k steps.

exp01 is the same config as Samuel's exp09, so it also checks that his result wasn't a lucky run.

In [ ]:
eps = "--eps-start 0.1 --eps-end 0.01 --eps-decay 50000"

runs = {
    "exp01": f"{eps} --lr 0.0001 --gamma 0.99 --batch-size 32",    # baseline (Samuel's exp09)
    "exp02": f"{eps} --lr 0.0005 --gamma 0.99 --batch-size 32",    # higher lr
    "exp03": f"{eps} --lr 0.001 --gamma 0.99 --batch-size 32",     # much higher lr
    "exp04": f"{eps} --lr 0.000025 --gamma 0.99 --batch-size 32",  # lower lr
    "exp05": f"{eps} --lr 0.0001 --gamma 0.95 --batch-size 32",    # lower gamma
    "exp06": f"{eps} --lr 0.0001 --gamma 0.9 --batch-size 32",     # much lower gamma
    "exp07": f"{eps} --lr 0.0001 --gamma 0.999 --batch-size 32",   # higher gamma
    "exp08": f"{eps} --lr 0.0001 --gamma 0.99 --batch-size 64",    # bigger batch
    "exp09": f"{eps} --lr 0.0001 --gamma 0.99 --batch-size 128",   # much bigger batch
    "exp10": f"{eps} --lr 0.0005 --gamma 0.99 --batch-size 64",    # higher lr + bigger batch
}

for name, run_args in runs.items():
    out_dir = f"/kaggle/working/Nkusi_Dan/{name}"
    # -u so the episode logs show up live instead of all at the end
    !uv run python -u train.py --total-steps 500000 {run_args} --out-dir {out_dir}
    if _exit_code != 0:
        raise RuntimeError(f"{name} failed, check the error above")
    !uv run python evaluate.py --model {out_dir}/dqn_model.pth --episodes 5 --out {out_dir}/eval.json
    if _exit_code != 0:
        raise RuntimeError(f"evaluating {name} failed, check the error above")

## Results

"Train" is the mean reward of the last 20 training episodes, and "greedy" is the mean of the 5 epsilon = 0 games.

In [ ]:
import csv
import json

print("run   | train  | greedy")
for name in runs:
    with open(f"/kaggle/working/Nkusi_Dan/{name}/log.csv") as f:
        rewards = [float(row["reward"]) for row in csv.DictReader(f)]
    with open(f"/kaggle/working/Nkusi_Dan/{name}/eval.json") as f:
        greedy = json.load(f)["eval_mean"]
    print(f"{name} | {sum(rewards[-20:]) / len(rewards[-20:]):6.1f} | {greedy:6.1f}")

In [ ]:
from IPython.display import Image, display

for name in runs:
    print(name)
    display(Image(f"/kaggle/working/Nkusi_Dan/{name}/plot.png"))